# 3. Reverse Process

The forward process destroys an image by adding noise. The reverse process *learns* to undo this — one step at a time. This notebook builds the training loop that makes diffusion models work.

By the end you will have:
- Understood the DDPM training objective (it's just MSE on noise predictions)
- Built a time-conditioned MLP denoiser in pure JAX
- Written a jit-compiled training step using `jax.value_and_grad`
- Trained on CIFAR-10 and verified that the model is learning to predict noise

In [69]:
# @title Setup — run this cell first (Colab only)
# !git clone https://github.com/maigimenez/let-it-rip
# %cd let-it-rip
# !pip install -q "jax[cuda12]" plotly jaxtyping optax datasets pillow

In [70]:
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import optax
import plotly.graph_objects as go
from datasets import load_dataset
from jaxtyping import Array, Float, Int, PRNGKeyArray
from plotly.subplots import make_subplots

print(f"JAX {jax.__version__} · backend: {jax.default_backend()}")

JAX 0.10.1 · backend: cpu


---
## Overview

In notebook 02 you derived the closed-form forward jump:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

and at the end noted the identity that connects the two directions:

$$\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\;\hat\varepsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}$$

This notebook builds $\hat\varepsilon_\theta$ — the denoiser network — and trains it.

In [71]:
# Carried forward from notebook 02 — these are given, not exercises.


def cosine_schedule(T: int = 1000, s: float = 0.008) -> dict:
    steps = jnp.arange(T + 1)
    alphas_bar_full = jnp.cos((steps / T + s) / (1 + s) * jnp.pi / 2) ** 2
    alphas_bar_full = alphas_bar_full / alphas_bar_full[0]
    betas = 1 - alphas_bar_full[1:] / alphas_bar_full[:-1]
    betas = jnp.clip(betas, 0, 0.999)
    alphas = 1.0 - betas
    return {"betas": betas, "alphas": alphas, "alphas_bar": alphas_bar_full[1:]}


def q_sample(
    x0: Float[Array, "h w c"],
    t: Int[Array, ""],
    noise: Float[Array, "h w c"],
    alphas_bar: Float[Array, " T"],
) -> Float[Array, "h w c"]:
    ab_t = alphas_bar[t]
    return jnp.sqrt(ab_t) * x0 + jnp.sqrt(1.0 - ab_t) * noise


T = 1000
schedule = cosine_schedule(T)

---
## Dataset — CIFAR-10

We switch from MNIST to CIFAR-10 (32 × 32 RGB) for the rest of the series. Colour images make generated outputs much easier to evaluate by eye.

One important change: we normalise pixel values to **[-1, 1]** instead of [0, 1]. This centres the distribution around zero, which stabilises training.

In [72]:
hf_ds = load_dataset("uoft-cs/cifar10", split="train[:5000]")
images = (
    jnp.array(
        np.stack([np.array(item["img"]) for item in hf_ds]),
        dtype=jnp.float32,
    )
    / 127.5
    - 1.0
)  # [0, 255] → [-1, 1]

print(
    f"images: {images.shape}  min={float(images.min()):.2f}  max={float(images.max()):.2f}"
)

images: (5000, 32, 32, 3)  min=-1.00  max=1.00


In [73]:
from PIL import Image as PILImage


def upscale(img_uint8, scale=4):
    h, w = img_uint8.shape[:2]
    pil = PILImage.fromarray(img_uint8).resize((w * scale, h * scale), PILImage.LANCZOS)
    return np.array(pil)


key = jr.PRNGKey(0)
idx = jr.randint(key, (32,), 0, len(images))
sample = np.clip((np.array(images[idx]) + 1) / 2, 0, 1)  # back to [0, 1] for display

fig = make_subplots(rows=4, cols=8, horizontal_spacing=0.01, vertical_spacing=0.01)
for i, img in enumerate(sample):
    r, c = divmod(i, 8)
    fig.add_trace(
        go.Image(z=upscale((img * 255).astype(np.uint8))), row=r + 1, col=c + 1
    )
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title="CIFAR-10 samples (normalised to [-1, 1] for training)", height=600
)
fig.show()

---
## 1 — The denoising objective

We want a network $\varepsilon_\theta(x_t, t)$ that predicts the noise $\varepsilon$ added to $x_0$ to produce $x_t$. The DDPM training objective simplifies to:

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t,\, x_0,\, \varepsilon}\!\left[\,\bigl\|\varepsilon - \varepsilon_\theta(x_t, t)\bigr\|^2\right]$$

This is just **mean-squared error between the true noise and the predicted noise**.

**Why predict $\varepsilon$ and not $x_0$?** You could instead train the network to predict the clean image directly. Predicting $\varepsilon$ works better in practice: the prediction target stays in a fixed range ($\varepsilon \sim \mathcal{N}(0,I)$ regardless of $t$), whereas predicting $x_0$ directly introduces timestep-dependent scaling that destabilises training at large $t$.

**The training loop** at each step is therefore:

1. Sample $x_0$ from the dataset
2. Sample $t \sim \text{Uniform}(1, T)$ and $\varepsilon \sim \mathcal{N}(0, I)$
3. Compute $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ &emsp;← notebook 02
4. Predict $\hat\varepsilon = \varepsilon_\theta(x_t, t)$
5. Loss $= \|\varepsilon - \hat\varepsilon\|^2$, backprop, update

---
## 2 — Time embeddings

The denoiser must know *which* timestep it is operating at: at $t \approx T$ the input is near-pure noise; at $t \approx 1$ it is almost clean. Passing the raw integer $t$ to the network works poorly — neural networks prefer smooth, high-dimensional inputs.

We represent $t$ as a continuous vector using **sinusoidal embeddings** — the same idea as positional encodings in the original Transformer, and exactly what DiT uses:

$$\text{emb}(t)_{2i} = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \qquad \text{emb}(t)_{2i+1} = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

for $i = 0, 1, \ldots, d/2 - 1$. This gives each timestep a unique fingerprint that varies smoothly across $t$.

### Exercise 1 — implement `sinusoidal_embedding`

Given a timestep `t` (integer) and embedding dimension `dim`, return a vector of shape `[dim]`.

The formula from the section header tells you exactly what to compute:

$$\text{emb}(t)_{2i} = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \qquad \text{emb}(t)_{2i+1} = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

Break it into three steps:

**Step 1: build the frequencies.**
You need $d/2$ values of $\tfrac{1}{10000^{2i/d}}$ for $i = 0, 1, \ldots, d/2 - 1$.
Taking the log turns the power into a product, which is easier to vectorise:

$$\frac{1}{10000^{2i/d}} = \exp\!\left(-\log(10000)\cdot \frac{i}{d/2 - 1}\right)$$

**Step 2: scale by `t`.**
Multiply every frequency by the timestep: `args = t * freqs` which gives a vector of $d/2$ angles.

**Step 3: concatenate sin and cos.**


In [74]:
def sinusoidal_embedding(
    t: int,
    dim: int = 256,
) -> Float[Array, " dim"]:
    # YOUR CODE HERE
    raise NotImplementedError

In [75]:
# @title 💡 Solution (open to reveal after trying to implement)


def sinusoidal_embedding(t: int, dim: int = 256) -> Float[Array, " dim"]:
    half = dim // 2
    freqs = jnp.exp(-jnp.log(10000) * jnp.arange(half, dtype=jnp.float32) / (half - 1))
    args = t * freqs
    return jnp.concatenate([jnp.sin(args), jnp.cos(args)])

In [76]:
emb_0 = sinusoidal_embedding(0)
emb_500 = sinusoidal_embedding(500)
emb_999 = sinusoidal_embedding(999)

assert emb_0.shape == (256,), f"expected (256,), got {emb_0.shape}"
assert not jnp.allclose(emb_0, emb_500), (
    "different timesteps must give different embeddings"
)
assert not jnp.allclose(emb_500, emb_999)
print("✓ sinusoidal_embedding works")

# Visualise: each row is an embedding for a different t
embs = jnp.stack([sinusoidal_embedding(t) for t in range(0, 1000, 10)])
fig = go.Figure(go.Heatmap(z=np.array(embs), colorscale="RdBu", zmid=0))
fig.update_layout(
    title="Sinusoidal time embeddings  (rows = timesteps 0..999, cols = embedding dims)",
    xaxis_title="embedding dim",
    yaxis_title="timestep t",
    height=350,
)
fig.show()

✓ sinusoidal_embedding works


---
## 3 — A simple MLP denoiser

Our first denoiser $\varepsilon_\theta$ is a small MLP. Later, we'll implement more realistic architecures, but the humble mlp is fast to train and keeps the focus on the *training loop* rather than the architecture.

```
  x_t [32, 32, 3]         t (int)
         │                    │
       flatten    sinusoidal_embedding
         │                    │
      [3072]               [256]
         │                    │
         │                FC(256)
         │                    │
         │                  Tanh
         │                    │
         └────── concat ───────┘
                     │
                  [3328]
                     │
                 FC(512)
                     │
                   GELU
                     │
                 FC(512)
                     │
                   GELU
                     │
                 FC(3072)
                     │
                  reshape
                     │
             [32, 32, 3]
```

In pure JAX a network is just a Python function plus a dictionary of parameters. Each linear layer is `{'W': ..., 'b': ...}`.


In [89]:
# FC network parameters are stored in a dict of dicts, with each layer having a weight matrix W and bias vector b.
# Activation functions (GELU, Tanh) are parameter-free, so they are not stored
# in params — they are applied directly in predict_noise.


def init_params(
    key: PRNGKeyArray,
    d_img: int = 3072,  # 32 * 32 * 3
    d_time: int = 256,
    d_hidden: int = 512,
) -> dict:
    k0, k1, k2, k3 = jr.split(key, 4)

    def linear(k, fan_in, fan_out):
        return {
            "W": jr.normal(k, (fan_in, fan_out)) * jnp.sqrt(2.0 / fan_in),
            "b": jnp.zeros(fan_out),
        }

    return {
        "time_proj": linear(k0, d_time, d_time),
        "fc1": linear(k1, d_img + d_time, d_hidden),
        "fc2": linear(k2, d_hidden, d_hidden),
        "fc3": linear(k3, d_hidden, d_img),
    }


key = jr.PRNGKey(0)
params = init_params(key)
print("parameter shapes:")
for name, layer in params.items():
    print(f"  {name}: W={layer['W'].shape}  b={layer['b'].shape}")


def linear(x, layer):
    return x @ layer["W"] + layer["b"]

parameter shapes:
  time_proj: W=(256, 256)  b=(256,)
  fc1: W=(3328, 512)  b=(512,)
  fc2: W=(512, 512)  b=(512,)
  fc3: W=(512, 3072)  b=(3072,)


### Exercise 2 — implement `predict_noise`

Given the parameter dictionary from `init_params`, a noisy image `xt` and a timestep `t`, return the predicted noise with the same shape as `xt`.

Steps:
1. Get the time embedding: call `sinusoidal_embedding(t)`, then apply the `time_proj` layer + `jnp.tanh`
2. Flatten `xt` to `[3072]` and concatenate with the projected embedding → `[3328]`
3. Forward through `fc1 → GELU → fc2 → GELU → fc3`
4. Reshape the output back to `xt.shape`

*Hint*: a linear layer is `x @ layer['W'] + layer['b']`. Use `jax.nn.gelu` for GELU.

In [78]:
def predict_noise(
    params: dict,
    xt: Float[Array, "h w c"],
    t: int,
) -> Float[Array, "h w c"]:
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


def predict_noise(
    params: dict,
    xt: Float[Array, "h w c"],
    t: int,
) -> Float[Array, "h w c"]:
    t_emb = sinusoidal_embedding(t)
    t_emb = jnp.tanh(linear(t_emb, params["time_proj"]))

    x = jnp.concatenate([xt.reshape(-1), t_emb])

    x = jax.nn.gelu(linear(x, params["fc1"]))
    x = jax.nn.gelu(linear(x, params["fc2"]))
    x = linear(x, params["fc3"])

    return x.reshape(xt.shape)

In [91]:
key = jr.PRNGKey(1)
xt_test = jr.normal(key, (32, 32, 3))

pred_t0 = predict_noise(params, xt_test, 0)
pred_t500 = predict_noise(params, xt_test, 500)

assert pred_t0.shape == (32, 32, 3), f"expected (32, 32, 3), got {pred_t0.shape}"
assert not jnp.allclose(pred_t0, pred_t500), "predictions must differ across timesteps"
print("✓ predict_noise works")

✓ predict_noise works


---
## 4 — The training step

Everything is now in place. Recall from notebook 01: `jax.value_and_grad(f)(x)` returns `(f(x), ∂f/∂x)` — we use this to get the loss and its gradient in one call.

For the optimiser we use **optax**, a pure-functional optimiser library. It follows the same pattern as JAX: state is carried explicitly, no mutation.

```python
optimizer = optax.adam(1e-4)
opt_state = optimizer.init(params)           # initialise state

# inside the training step:
loss, grads   = jax.value_and_grad(loss_fn)(params)
updates, opt_state = optimizer.update(grads, opt_state)
params        = optax.apply_updates(params, updates)
```

In [ ]:
optimizer = optax.adam(learning_rate=1e-4)
opt_state = optimizer.init(params)

### Exercise 3 — implement `train_step`

This is the central exercise of the notebook. The function takes the current `params` and `opt_state`, a batch of clean images, a PRNG key, and the noise schedule, and returns updated versions plus the scalar loss.

Inside, you need to:
1. Split the key and sample random timesteps `ts` (one per image) and noise
2. Define `loss_fn(params)`:
   - Compute `xt_batch` by vmapping `q_sample` over the batch
   - Predict noise by vmapping `predict_noise` over the batch
   - Return the mean squared error
3. Call `jax.value_and_grad(loss_fn)(params)` to get loss and gradients
4. Apply the optimiser update and return updated `(params, opt_state, loss)`

*Hint*: for `vmap(q_sample, in_axes=(0, 0, 0, None))` — the schedule array is shared across the batch (`None`).

In [82]:
def train_step(
    params: dict,
    opt_state: optax.OptState,
    x0_batch: Float[Array, "b h w c"],
    key: PRNGKeyArray,
    schedule: dict,
) -> tuple[dict, optax.OptState, Float[Array, ""]]:
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


def train_step(
    params: dict,
    opt_state: optax.OptState,
    x0_batch: Float[Array, "b h w c"],
    key: PRNGKeyArray,
    schedule: dict,
) -> tuple[dict, optax.OptState, Float[Array, ""]]:
    batch = x0_batch.shape[0]
    key_t, key_noise = jr.split(key)

    ts = jr.randint(key_t, (batch,), 0, T)
    noise = jr.normal(key_noise, x0_batch.shape)

    def loss_fn(params):
        xt_batch = jax.vmap(q_sample, in_axes=(0, 0, 0, None))(
            x0_batch, ts, noise, schedule["alphas_bar"]
        )
        eps_pred = jax.vmap(predict_noise, in_axes=(None, 0, 0))(params, xt_batch, ts)
        return jnp.mean((noise - eps_pred) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss

In [96]:
# Sanity-check: loss should go down over a few steps on the same batch
key = jr.PRNGKey(42)
key, batch_key = jr.split(key)
small_batch = images[:16]

train_step_jit = jax.jit(train_step)

# Warm-up (triggers compilation, first call is slower)
params_v, opt_state_v, _ = train_step_jit(params, opt_state, small_batch, key, schedule)

losses_check = []
for _ in range(20):
    key, step_key = jr.split(key)
    params_v, opt_state_v, loss = train_step_jit(
        params_v, opt_state_v, small_batch, step_key, schedule
    )
    losses_check.append(float(loss))

assert losses_check[-1] < losses_check[0], "loss should decrease over 20 steps"
print(f"✓ train_step works  (loss {losses_check[0]:.4f} → {losses_check[-1]:.4f})")

✓ train_step works  (loss 1.8568 → 1.1574)


---
## 5 — Training on CIFAR-10

The hard parts are done! The training loop below is given, read through it once to confirm your understaning of the pseudocode in section 1, then run it.

In [ ]:
def train(params, opt_state, images, key, n_epochs=15, batch_size=128):
    n = len(images)
    history = []

    for epoch in range(1, n_epochs + 1):
        key, shuffle_key = jr.split(key)
        perm = jr.permutation(shuffle_key, n)
        imgs_shuffled = images[perm]

        epoch_losses = []
        for i in range(0, n - batch_size + 1, batch_size):
            batch = imgs_shuffled[i : i + batch_size]
            key, step_key = jr.split(key)
            params, opt_state, loss = train_step_jit(
                params, opt_state, batch, step_key, schedule
            )
            epoch_losses.append(float(loss))

        mean_loss = float(np.mean(epoch_losses))
        history.append(mean_loss)
        print(f"epoch {epoch:2d}/{n_epochs}  loss = {mean_loss:.4f}")

    return params, opt_state, history


# Re-initialise so training starts from scratch
key = jr.PRNGKey(0)
params = init_params(key)
opt_state = optimizer.init(params)
train_step_jit = jax.jit(train_step)

key, train_key = jr.split(key)
params, opt_state, history = train(params, opt_state, images, train_key)

In [86]:
fig = go.Figure(go.Scatter(y=history, mode="lines+markers", name="train loss"))
fig.update_layout(
    title="Training loss",
    xaxis_title="epoch",
    yaxis_title="MSE",
    height=350,
)
fig.show()

### Noise prediction quality

A well-trained model should:
- At **large $t$** (near pure noise): predict roughly the full noise added
- At **small $t$** (near clean image): predict a small correction

The grid below shows five timesteps for one CIFAR-10 image: original, noisy input, true noise, and predicted noise.

In [87]:
key = jr.PRNGKey(7)
img = images[0]  # one image

timesteps_viz = [50, 200, 400, 700, 950]
n_cols = len(timesteps_viz)

rows_labels = ["x₀ (original)", "xₜ (noisy)", "ε (true noise)", "ε̂ (predicted)"]
fig = make_subplots(
    rows=4,
    cols=n_cols,
    subplot_titles=[f"t = {t}" for t in timesteps_viz] + [""] * (3 * n_cols),
    horizontal_spacing=0.02,
    vertical_spacing=0.04,
)


def to_uint8(arr):
    arr = np.array(arr)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    return (arr * 255).astype(np.uint8)


for col, t in enumerate(timesteps_viz, start=1):
    key, noise_key = jr.split(key)
    noise = jr.normal(noise_key, img.shape)
    xt = q_sample(img, t, noise, schedule["alphas_bar"])
    eps_pred = predict_noise(params, xt, t)

    # original (clipped to [-1,1] display range)
    fig.add_trace(go.Image(z=to_uint8(img)), row=1, col=col)
    fig.add_trace(go.Image(z=to_uint8(xt)), row=2, col=col)
    fig.add_trace(go.Image(z=to_uint8(noise)), row=3, col=col)
    fig.add_trace(go.Image(z=to_uint8(eps_pred)), row=4, col=col)

# Row labels on the left
for row, label in enumerate(rows_labels, start=1):
    fig.add_annotation(
        x=-0.01,
        y=(4 - row + 0.5) / 4,
        xref="paper",
        yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        xanchor="right",
        font=dict(size=11),
    )

fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(title="Noise predictions across timesteps", height=500)
fig.show()

In [88]:
# Save trained parameters so notebook 04 can load them directly.
import pickle

with open("params_nb03.pkl", "wb") as f:
    pickle.dump(params, f)

print("Saved → params_nb03.pkl")

Saved → params_nb03.pkl


---
## What's next?

You have a trained denoiser $\varepsilon_\theta$. In **04. Sampling** you will use it to *generate* new CIFAR-10 images by running the reverse process: start from pure noise $x_T \sim \mathcal{N}(0, I)$ and iteratively apply:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t, t)\right) + \sqrt{\tilde\beta_t}\,z, \qquad z \sim \mathcal{N}(0, I)$$

You will also implement the **DDIM sampler** — the same denoiser, deterministic output, 50 steps instead of 1000.